In [1]:
import numpy as np
import pandas as pd
from paraphrase_detection import data_shuffle_split, SBERTPairClassifier, Train
import torch
from sentence_transformers import SentenceTransformer
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LinearLR

/Users/oli.dmrs/Documents/Personal Projects/paraphrase-detection-ml/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_parquet("hf://datasets/sentence-transformers/quora-duplicates/pair-class/train-00000-of-00001.parquet")
data = df.to_numpy()
data = data[:5000]

X = data[:, :2]
y = data[:, 2:3]

X_train, X_val, X_test, y_train, y_val, y_test = data_shuffle_split(X, y, 0.15, 0.12, 42)
training_data = np.hstack((X_train, y_train))
validation_data = np.hstack((X_val, y_val))
test_data = np.hstack((X_test, y_test))

In [13]:
model = SentenceTransformer("all-MiniLM-L6-v2")
fc_layer_sizes = [model.get_sentence_embedding_dimension()]
dropout_p = 0.1
lr = 1e-3
epochs = 10
batch_size = 32
steps = epochs * batch_size
n_warmup_steps = steps * 0.1
n_freeze = 3

model = SBERTPairClassifier(model, fc_layer_sizes, dropout_p)
optimizer = torch.optim.AdamW(model.parameters(), lr)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = n_warmup_steps, num_training_steps = steps)
criterion = torch.nn.CrossEntropyLoss()
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

In [ ]:
train_loader = DataLoader(dataset = training_data, batch_size = batch_size, shuffle = True, num_workers = 4)
validation_loader = DataLoader(dataset = validation_data, batch_size = batch_size, shuffle = True, num_workers = 4)
test_loader = DataLoader(dataset = test_data, batch_size = batch_size, shuffle = True, num_workers = 4)

TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.

In [ ]:
train = Train(model = model,
              optimizer = optimizer,
              scheduler = scheduler,
              criterion = criterion, 
              device = device,
              n_freeze = n_freeze,
              epochs = epochs,
              train_dataloader = train_loader,
              val_dataloader = validation_loader
              )

best_params, avg_batch_train_loss, epoch_train_acc, avg_batch_val_loss, epoch_val_acc, epoch_val_f1 = train.run_training_loop()
